<a href="https://colab.research.google.com/github/SanviSaparia/Google-Colab/blob/main/chapter_appendix-tools-for-deep-learning/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy matplotlib scikit-learn nltk tensorflow

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, GRU, LSTM, Dense,
    Attention, AdditiveAttention, Concatenate
)

from sklearn.model_selection import train_test_split

from nltk.translate.bleu_score import sentence_bleu
import nltk

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
!wget https://raw.githubusercontent.com/SayamAlt/English-to-French-Language-Translation-using-Seq2Seq-Modeling/main/eng-french.csv -O eng-fra.csv

--2026-10-01 05:28:30--  https://raw.githubusercontent.com/SayamAlt/English-to-French-Language-Translation-using-Seq2Seq-Modeling/main/eng-french.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12497909 (12M) [text/plain]
Saving to: ‘eng-fra.csv’

eng-fra.csv         100%[===================>]  11.92M  --.-KB/s    in 0.08s   

2026-10-01 05:28:30 (144 MB/s) - ‘eng-fra.csv’ saved [12497909/12497909]



In [5]:
df = pd.read_csv(
    "eng-fra.csv",
    header=None,
    names=["english", "french"]
)

print(df.head())
print(df.shape)

                   english                  french
0  English words/sentences  French words/sentences
1                      Hi.                  Salut!
2                     Run!                 Cours !
3                     Run!                Courez !
4                     Who?                   Qui ?
(175622, 2)


In [6]:
df["english"] = df["english"].astype(str).str.lower().str.strip()
df["french"] = df["french"].astype(str).str.lower().str.strip()

df.head()

df["french"] = df["french"].apply(
    lambda x: "<start> " + x + " <end>"
)

df.head()

,english,french
0,english words/sentences,<start> french words/sentences <end>
1,hi.,<start> salut! <end>
2,run!,<start> cours ! <end>
3,run!,<start> courez ! <end>
4,who?,<start> qui ? <end>


In [7]:
eng_tokenizer = Tokenizer(
    num_words=10000,
    filters=""
)

eng_tokenizer.fit_on_texts(df["english"])

eng_sequences = eng_tokenizer.texts_to_sequences(
    df["english"]
)

print(eng_sequences[:2])

[[457], [5297]]


In [8]:
fra_tokenizer = Tokenizer(
    num_words=10000,
    filters=""
)

fra_tokenizer.fit_on_texts(df["french"])

fra_sequences = fra_tokenizer.texts_to_sequences(
    df["french"]
)

print(fra_sequences[:2])

[[1, 2], [1, 2]]


In [9]:
max_eng_len = max(
    len(sequence) for sequence in eng_sequences
)

max_fra_len = max(
    len(sequence) for sequence in fra_sequences
)

print("English max length:", max_eng_len)
print("French max length:", max_fra_len)

eng_sequences = pad_sequences(
    eng_sequences,
    maxlen=max_eng_len,
    padding="post"
)

fra_sequences = pad_sequences(
    fra_sequences,
    maxlen=max_fra_len,
    padding="post"
)

print(eng_sequences.shape)
print(fra_sequences.shape)

English max length: 43
French max length: 54
(175622, 43)
(175622, 54)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    eng_sequences,
    fra_sequences,
    test_size=0.20,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (140497, 43)
Testing: (35125, 43)


In [11]:
embedding_dim = 128
hidden_units = 256

eng_vocab_size = min(
    10000,
    len(eng_tokenizer.word_index) + 1
)

fra_vocab_size = min(
    10000,
    len(fra_tokenizer.word_index) + 1
)

encoder_inputs = Input(
    shape=(None,),
    name="encoder_inputs"
)

encoder_embedding = Embedding(
    eng_vocab_size,
    embedding_dim,
    mask_zero=True
)(encoder_inputs)

encoder_gru = GRU(
    hidden_units,
    return_sequences=True,
    return_state=True
)

encoder_outputs, encoder_state = encoder_gru(
    encoder_embedding
)

In [12]:
decoder_inputs = Input(
    shape=(None,),
    name="decoder_inputs"
)

decoder_embedding = Embedding(
    fra_vocab_size,
    embedding_dim,
    mask_zero=True
)(decoder_inputs)

decoder_gru = GRU(
    hidden_units,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _ = decoder_gru(
    decoder_embedding,
    initial_state=encoder_state
)

soft_attention = AdditiveAttention(
    name="soft_attention"
)

context = soft_attention(
    [decoder_outputs, encoder_outputs]
)

combined = Concatenate(axis=-1)(
    [decoder_outputs, context]
)

soft_output = Dense(
    fra_vocab_size,
    activation="softmax"
)(combined)

soft_model = Model(
    [encoder_inputs, decoder_inputs],
    soft_output
)

soft_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

soft_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │  1,280,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │  1,280,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ [(None, None,     │    296,448 │ embedding[0][0],  │
│                     │ 256), (None,      │            │ not_equal[0][0]   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, None,     │    296,448 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ gru[0][1]         │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ soft_attention      │ (None, None, 256) │        256 │ gru_1[0][0],      │
│ (AdditiveAttention) │                   │            │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None, 512) │          0 │ gru_1[0][0],      │
│ (Concatenate)       │                   │            │ soft_attention[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  5,130,000 │ concatenate[0][0] │
│                     │ 10000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,283,152 (31.60 MB)

 Trainable params: 8,283,152 (31.60 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
decoder_input_train = y_train[:, :-1]
decoder_target_train = y_train[:, 1:]

decoder_input_test = y_test[:, :-1]
decoder_target_test = y_test[:, 1:]

soft_history = soft_model.fit(
    [X_train, decoder_input_train],
    np.expand_dims(decoder_target_train, -1),
    validation_split=0.1,
    epochs=10,
    batch_size=64
)

Epoch 1/10
 184/1976 ━━━━━━━━━━━━━━━━━━━━ 3:34:49 7s/step - accuracy: 0.2178 - loss: 6.6383

In [ ]:
global_attention = Attention(
    name="global_attention"
)

global_context = global_attention(
    [decoder_outputs, encoder_outputs]
)

global_combined = Concatenate(axis=-1)(
    [decoder_outputs, global_context]
)

global_output = Dense(
    fra_vocab_size,
    activation="softmax"
)(global_combined)

global_model = Model(
    [encoder_inputs, decoder_inputs],
    global_output
)

global_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

global_model.summary()

In [ ]:
global_history = global_model.fit(
    [X_train, decoder_input_train],
    np.expand_dims(decoder_target_train, -1),
    validation_split=0.1,
    epochs=10,
    batch_size=64
)

In [ ]:
cross_attention = Attention(
    name="cross_attention"
)

cross_context = cross_attention(
    [decoder_outputs, encoder_outputs]
)

cross_combined = Concatenate(axis=-1)(
    [decoder_outputs, cross_context]
)

cross_output = Dense(
    fra_vocab_size,
    activation="softmax"
)(cross_combined)

cross_model = Model(
    [encoder_inputs, decoder_inputs],
    cross_output
)

cross_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cross_model.summary()

In [ ]:
cross_history = cross_model.fit(
    [X_train, decoder_input_train],
    np.expand_dims(decoder_target_train, -1),
    validation_split=0.1,
    epochs=10,
    batch_size=64
)

hard_scores = tf.keras.layers.Dot(
    axes=-1,
    normalize=True,
    name="hard_scores"
)([decoder_outputs, encoder_outputs])

hard_index = tf.argmax(
    hard_scores,
    axis=-1
)

hard_context = tf.reduce_max(
    hard_scores,
    axis=-1,
    keepdims=True
)

hard_combined = Concatenate(axis=-1)(
    [decoder_outputs, hard_context]
)

hard_output = Dense(
    fra_vocab_size,
    activation="softmax"
)(hard_combined)

In [ ]:
reverse_fra_vocab = {
    value: key
    for key, value in fra_tokenizer.word_index.items()
}

def sequence_to_sentence(sequence):
    words = []

    for index in sequence:
        if index == 0:
            continue

        word = reverse_fra_vocab.get(index, "")

        if word == "<start>":
            continue

        if word == "<end>":
            break

        words.append(word)

    return " ".join(words)

In [ ]:
def calculate_bleu(model, X_data, y_data, number_of_samples=10):

    scores = []

    for i in range(
        min(number_of_samples, len(X_data))
    ):

        decoder_input = y_data[i:i+1, :-1]

        prediction = model.predict(
            [X_data[i:i+1], decoder_input],
            verbose=0
        )

        predicted_ids = np.argmax(
            prediction[0],
            axis=-1
        )

        reference = sequence_to_sentence(
            y_data[i]
        ).split()

        candidate = sequence_to_sentence(
            predicted_ids
        ).split()

        if len(candidate) == 0:
            continue

        score = sentence_bleu(
            [reference],
            candidate
        )

        scores.append(score)

    return np.mean(scores)

In [ ]:
soft_bleu = calculate_bleu(
    soft_model,
    X_test,
    y_test
)

global_bleu = calculate_bleu(
    global_model,
    X_test,
    y_test
)

cross_bleu = calculate_bleu(
    cross_model,
    X_test,
    y_test
)

print("Soft Attention BLEU:", soft_bleu)
print("Global Attention BLEU:", global_bleu)
print("Cross Attention BLEU:", cross_bleu)

In [ ]:
results = pd.DataFrame({
    "Attention": [
        "Soft",
        "Global",
        "Cross"
    ],
    "BLEU Score": [
        soft_bleu,
        global_bleu,
        cross_bleu
    ]
})

results

In [ ]:
import random

random_indices = random.sample(
    range(len(X_test)),
    10
)

for i in random_indices:

    print("=" * 60)

    english_sentence = " ".join(
        eng_tokenizer.sequences_to_texts(
            [X_test[i]]
        )[0].split()
    )

    actual_french = sequence_to_sentence(
        y_test[i]
    )

    print("English:")
    print(english_sentence)

    print("\nActual French:")
    print(actual_french)

    for name, model in [
        ("Soft Attention", soft_model),
        ("Global Attention", global_model),
        ("Cross Attention", cross_model)
    ]:

        prediction = model.predict(
            [
                X_test[i:i+1],
                y_test[i:i+1, :-1]
            ],
            verbose=0
        )

        predicted_ids = np.argmax(
            prediction[0],
            axis=-1
        )

        predicted_sentence = sequence_to_sentence(
            predicted_ids
        )

        print("\n" + name + ":")
        print(predicted_sentence)

In [ ]:
attention_layer = soft_model.get_layer(
    "soft_attention"
)

attention_model = Model(
    soft_model.inputs,
    [
        soft_model.output,
        attention_layer.output
    ]
)

sample_index = 0

predictions, attention_weights = attention_model.predict(
    [
        X_test[sample_index:sample_index+1],
        y_test[sample_index:sample_index+1, :-1]
    ],
    verbose=0
)

print(attention_weights.shape)

plt.figure(figsize=(12, 8))

plt.imshow(
    attention_weights[0],
    aspect="auto"
)

plt.xlabel("Input English Words")
plt.ylabel("Output French Words")

plt.title("Soft Attention Heatmap")

plt.colorbar()

plt.show()

In [ ]:
global_attention_layer = global_model.get_layer(
    "global_attention"
)

global_attention_model = Model(
    global_model.inputs,
    [
        global_model.output,
        global_attention_layer.output
    ]
)

predictions, global_weights = global_attention_model.predict(
    [
        X_test[0:1],
        y_test[0:1, :-1]
    ],
    verbose=0
)

print(global_weights.shape)

plt.figure(figsize=(12, 8))

plt.imshow(
    global_weights[0],
    aspect="auto"
)

plt.xlabel("Input English Words")
plt.ylabel("Output French Words")

plt.title("Global Attention Heatmap")

plt.colorbar()

plt.show()

In [ ]:
cross_attention_layer = cross_model.get_layer(
    "cross_attention"
)

cross_attention_model = Model(
    cross_model.inputs,
    [
        cross_model.output,
        cross_attention_layer.output
    ]
)

predictions, cross_weights = cross_attention_model.predict(
    [
        X_test[0:1],
        y_test[0:1, :-1]
    ],
    verbose=0
)

print(cross_weights.shape)

plt.figure(figsize=(12, 8))

plt.imshow(
    cross_weights[0],
    aspect="auto"
)

plt.xlabel("Input English Words")
plt.ylabel("Output French Words")

plt.title("Cross Attention Heatmap")

plt.colorbar()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    soft_history.history["loss"],
    label="Soft Attention"
)

plt.plot(
    global_history.history["loss"],
    label="Global Attention"
)

plt.plot(
    cross_history.history["loss"],
    label="Cross Attention"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training Loss Comparison")

plt.legend()

plt.show()